# Data Sources: First Visuals

A first look at the three public data sources for this project. For each source we pull a small sample directly from its REST API and plot it as a simple time series, just to see the shape of the data.

| Data | Source | Docs |
|---|---|---|
| River flow | USGS Water Data OGC API | https://api.waterdata.usgs.gov/ogcapi/v0/ |
| Reservoir elevation | Bureau of Reclamation RISE API | https://data.usbr.gov/rise-api |
| Snowpack (SNOTEL) | USDA NRCS AWDB REST API | https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/index.html |

**Note on the RISE API:** at the time this notebook was written, `data.usbr.gov` was intermittently timing out. The RISE cell below catches that specific failure so the rest of the notebook still runs — re-run it later if it's empty.


In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import requests

from colorado_river_viz import (
    LAKE_MEAD,
    LAKE_POWELL,
    DateRange,
    fetch_rise_time_series,
    fetch_snotel_daily_values,
    fetch_usgs_daily_values,
)

%matplotlib widget

years = 100
DATE_RANGE = DateRange(
    start=date.today() - timedelta(days=years * 365), end=date.today()
)

## River flow: four gauges along the mainstem (USGS)

Four USGS gauges tracing discharge from above Lake Powell to below Hoover Dam. Parameter `00060` is discharge in cubic feet per second.

| Gauge | Site ID | Position |
|---|---|---|
| Colorado River near Cisco, UT | `09180500` | Just above Lake Powell |
| Colorado River at Lees Ferry, AZ | `09380000` | Just below Glen Canyon Dam, marking the Colorado River Compact's Lee Ferry point |
| Colorado River above Diamond Creek near Peach Springs, AZ | `09404200` | Just above Lake Mead |
| Colorado River below Hoover Dam, AZ-NV | `09421500` | Just below Hoover Dam |

Source: https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily


In [ ]:
cisco_flow = fetch_usgs_daily_values(
    monitoring_location_id="USGS-09180500",
    parameter_code="00060",
    date_range=DATE_RANGE,
)
lees_ferry_flow = fetch_usgs_daily_values(
    monitoring_location_id="USGS-09380000",
    parameter_code="00060",
    date_range=DATE_RANGE,
)
diamond_creek_flow = fetch_usgs_daily_values(
    monitoring_location_id="USGS-09404200",
    parameter_code="00060",
    date_range=DATE_RANGE,
)
below_hoover_flow = fetch_usgs_daily_values(
    monitoring_location_id="USGS-09421500",
    parameter_code="00060",
    date_range=DATE_RANGE,
)
lees_ferry_flow.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(
    cisco_flow.index,
    cisco_flow["value"],
    color="#0072B2",
    linewidth=1,
    label="Above Lake Powell (Cisco, UT — 09180500)",
)
ax.plot(
    lees_ferry_flow.index,
    lees_ferry_flow["value"],
    color="#E69F00",
    linewidth=1,
    label="Below Glen Canyon Dam (Lees Ferry, AZ — 09380000)",
)
ax.plot(
    diamond_creek_flow.index,
    diamond_creek_flow["value"],
    color="#009E73",
    linewidth=1,
    label="Above Lake Mead (Diamond Creek, AZ — 09404200)",
)
ax.plot(
    below_hoover_flow.index,
    below_hoover_flow["value"],
    color="#D55E00",
    linewidth=1,
    label="Below Hoover Dam, AZ-NV — 09421500",
)
ax.set_title("Colorado River discharge: above Lake Powell to below Hoover Dam")
ax.set_xlabel("Date")
ax.set_ylabel("Discharge (cubic feet per second)")
ax.legend()
fig.tight_layout()

## Reservoir storage & elevation: Lake Powell & Lake Mead (Bureau of Reclamation RISE)

Two views of the same two reservoirs:

- **Storage as a percent of full-pool capacity** — the most intuitive "how full is it" number. Lake Powell's operational full-pool capacity is 24,300,000 acre-feet; Lake Mead's is 26,120,000 acre-feet (RISE catalog items `509` and `6124`).
- **Elevation against three reference points** — reservoir elevation (items `508` and `6123`) against full pool, and the elevation below which each dam can no longer generate hydropower ("minimum power pool") and below which water can no longer be released by gravity at all ("dead pool").

| Reservoir | Full pool | Minimum power pool | Dead pool |
|---|---|---|---|
| Lake Powell (Glen Canyon Dam) | 3,700 ft | 3,490 ft | 3,374 ft |
| Lake Mead (Hoover Dam) | 1,219.64 ft | 950 ft | 895 ft |

Figures and sources: [`src/colorado_river_viz/reservoirs.py`](../src/colorado_river_viz/reservoirs.py).

Source: https://data.usbr.gov/rise-api


In [ ]:
try:
    powell_elevation = fetch_rise_time_series(
        catalog_item_id=LAKE_POWELL.elevation_catalog_item_id, date_range=DATE_RANGE
    )
    powell_storage = fetch_rise_time_series(
        catalog_item_id=LAKE_POWELL.storage_catalog_item_id, date_range=DATE_RANGE
    )
    mead_elevation = fetch_rise_time_series(
        catalog_item_id=LAKE_MEAD.elevation_catalog_item_id, date_range=DATE_RANGE
    )
    mead_storage = fetch_rise_time_series(
        catalog_item_id=LAKE_MEAD.storage_catalog_item_id, date_range=DATE_RANGE
    )
except requests.exceptions.RequestException as exc:
    print(
        f"RISE API request failed ({exc}); "
        "skipping the reservoir storage/elevation plots for now."
    )
    powell_elevation = None
    powell_storage = None
    mead_elevation = None
    mead_storage = None

In [ ]:
if powell_storage is not None and mead_storage is not None:
    powell_percent_full = (
        100 * powell_storage["result"] / LAKE_POWELL.full_pool_capacity_af
    )
    mead_percent_full = 100 * mead_storage["result"] / LAKE_MEAD.full_pool_capacity_af

    powell_min_power_pool_percent = (
        100
        * LAKE_POWELL.minimum_power_pool_storage_af
        / LAKE_POWELL.full_pool_capacity_af
    )
    mead_min_power_pool_percent = (
        100 * LAKE_MEAD.minimum_power_pool_storage_af / LAKE_MEAD.full_pool_capacity_af
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(
        powell_percent_full.index,
        powell_percent_full,
        color="#0072B2",
        linewidth=1,
        label="Lake Powell",
    )
    ax.plot(
        mead_percent_full.index,
        mead_percent_full,
        color="#E69F00",
        linewidth=1,
        label="Lake Mead",
    )
    ax.axhline(
        powell_min_power_pool_percent,
        color="#0072B2",
        linewidth=1,
        linestyle="--",
        label="Lake Powell minimum power pool",
    )
    ax.axhline(
        mead_min_power_pool_percent,
        color="#E69F00",
        linewidth=1,
        linestyle="--",
        label="Lake Mead minimum power pool",
    )
    # Dead pool is the zero point these percent-full figures are referenced against
    # (see reservoirs.py), so it's the same 0% line for both reservoirs.
    ax.axhline(
        0,
        color="#D55E00",
        linewidth=1,
        linestyle=":",
        label="Dead pool (both reservoirs)",
    )
    ax.set_title("Reservoir storage as a percent of full-pool capacity")
    ax.set_xlabel("Date")
    ax.set_ylabel("Percent of full-pool capacity")
    ax.set_ylim(0, 100)
    ax.legend()
    fig.tight_layout()

In [ ]:
if powell_elevation is not None and mead_elevation is not None:
    fig, (ax_powell, ax_mead) = plt.subplots(
        nrows=2, ncols=1, figsize=(9, 6), sharex=True
    )
    ax_powell.plot(
        powell_elevation.index, powell_elevation["result"], color="#0072B2", linewidth=1
    )
    ax_powell.axhline(
        LAKE_POWELL.full_pool_elevation_ft,
        color="#009E73",
        linewidth=1,
        linestyle="--",
        label="Full pool",
    )
    ax_powell.axhline(
        LAKE_POWELL.minimum_power_pool_ft,
        color="#E69F00",
        linewidth=1,
        linestyle="--",
        label="Minimum power pool",
    )
    ax_powell.axhline(
        LAKE_POWELL.dead_pool_ft,
        color="#D55E00",
        linewidth=1,
        linestyle="--",
        label="Dead pool",
    )
    ax_powell.set_title("Lake Powell elevation vs. critical cutoffs (Glen Canyon Dam)")
    ax_powell.set_ylabel("Elevation (feet)")
    ax_powell.legend()

    ax_mead.plot(
        mead_elevation.index, mead_elevation["result"], color="#0072B2", linewidth=1
    )
    ax_mead.axhline(
        LAKE_MEAD.full_pool_elevation_ft,
        color="#009E73",
        linewidth=1,
        linestyle="--",
        label="Full pool",
    )
    ax_mead.axhline(
        LAKE_MEAD.minimum_power_pool_ft,
        color="#E69F00",
        linewidth=1,
        linestyle="--",
        label="Minimum power pool",
    )
    ax_mead.axhline(
        LAKE_MEAD.dead_pool_ft,
        color="#D55E00",
        linewidth=1,
        linestyle="--",
        label="Dead pool",
    )
    ax_mead.set_title("Lake Mead elevation vs. critical cutoffs (Hoover Dam)")
    ax_mead.set_xlabel("Date")
    ax_mead.set_ylabel("Elevation (feet)")
    ax_mead.legend()

    fig.tight_layout()

## Snowpack: SNOTEL snow water equivalent (USDA NRCS AWDB)

Two stations feeding different headwaters of the Colorado: Berthoud Summit (`335:CO:SNTL`, Upper Colorado headwaters near Rocky Mountain National Park) and Red Mountain Pass (`713:CO:SNTL`, San Juan Mountains). Element `WTEQ` is snow water equivalent in inches.

Source: https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/index.html


In [ ]:
berthoud_summit_swe = fetch_snotel_daily_values(
    station_triplet="335:CO:SNTL", element_code="WTEQ", date_range=DATE_RANGE
)
red_mountain_pass_swe = fetch_snotel_daily_values(
    station_triplet="713:CO:SNTL", element_code="WTEQ", date_range=DATE_RANGE
)
berthoud_summit_swe.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(
    berthoud_summit_swe.index,
    berthoud_summit_swe["value"],
    color="#0072B2",
    linewidth=1,
    label="Berthoud Summit (335:CO:SNTL)",
)
ax.plot(
    red_mountain_pass_swe.index,
    red_mountain_pass_swe["value"],
    color="#E69F00",
    linewidth=1,
    label="Red Mountain Pass (713:CO:SNTL)",
)
ax.set_title("Snow water equivalent at two Colorado SNOTEL stations")
ax.set_xlabel("Date")
ax.set_ylabel("Snow water equivalent (inches)")
ax.legend()
fig.tight_layout()